# RAGAS Core Metrics -- Complete RAG Evaluation

This notebook covers the core evaluation metrics provided by the **RAGAS** (Retrieval Augmented Generation Assessment) framework.

RAGAS provides research-backed metrics specifically designed for evaluating RAG pipelines. In this notebook we will:

1. Build a RAG pipeline using OpenAI + Qdrant (in-memory)
2. Create a RAGAS `EvaluationDataset` with `SingleTurnSample` objects
3. Evaluate using **Faithfulness**, **Answer Relevancy**, **Context Precision**, **Context Recall**, **Context Entity Recall**, and **Answer Correctness**
4. Run all metrics together and analyze results

**Prerequisites**: Notebooks 01-05 completed, OpenAI API key set in `.env`

## 1. Installation and Setup

In [ ]:
# Install required packages (if not already installed)
# !pip install ragas openai qdrant-client pandas python-dotenv

In [ ]:
import os
import json
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv(os.path.join("..", ".env"))

# Verify API key is set
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("API key loaded successfully.")

In [ ]:
import openai
import pandas as pd
import numpy as np
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# RAGAS imports
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ContextEntityRecall,
    FactualCorrectness,
    SemanticSimilarity,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print("All imports successful.")

## 2. Build the RAG Pipeline

We will build a complete RAG pipeline with:
- **Knowledge base**: Documents about a fictional e-commerce company "Acme Corp"
- **Vector store**: Qdrant (in-memory)
- **Embeddings**: OpenAI text-embedding-3-small
- **Generator**: OpenAI GPT-4o-mini

In [ ]:
# Initialize clients
openai_client = OpenAI()
qdrant_client = QdrantClient(":memory:")

EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-4o-mini"
COLLECTION_NAME = "acme_corp_docs"
EMBEDDING_DIM = 1536

print(f"Using embedding model: {EMBEDDING_MODEL}")
print(f"Using generation model: {GENERATION_MODEL}")

In [ ]:
# Acme Corp knowledge base -- fictional e-commerce company documents
ACME_DOCUMENTS = [
    {
        "id": 1,
        "title": "Return Policy Overview",
        "content": (
            "Acme Corp offers a 30-day return policy on all products purchased through "
            "our website or retail stores. To be eligible for a return, the item must be "
            "unused, in its original packaging, and accompanied by the original receipt or "
            "proof of purchase. Refunds are processed within 5-7 business days after we "
            "receive the returned item. Shipping costs for returns are the responsibility "
            "of the customer unless the item was defective or the wrong item was shipped. "
            "Items marked as 'Final Sale' cannot be returned or exchanged."
        ),
    },
    {
        "id": 2,
        "title": "Electronics Return Policy",
        "content": (
            "Electronic products purchased from Acme Corp have a 15-day return window "
            "instead of the standard 30 days. All electronics must be returned with all "
            "original accessories, cables, manuals, and packaging. A restocking fee of 15% "
            "may apply to opened electronics. Defective electronics can be exchanged for "
            "the same model within the first 90 days of purchase at no additional cost. "
            "Software products and digital downloads are non-returnable once the seal is "
            "broken or the download code has been redeemed."
        ),
    },
    {
        "id": 3,
        "title": "Shipping Policy",
        "content": (
            "Acme Corp offers several shipping options for domestic orders within the "
            "United States. Standard Shipping takes 5-7 business days and is free for "
            "orders over $50. Expedited Shipping takes 2-3 business days and costs $12.99. "
            "Overnight Shipping is available for $24.99 and delivers the next business day "
            "if ordered before 2 PM EST. Orders are processed within 1-2 business days. "
            "Tracking information is sent via email once the package has been shipped."
        ),
    },
    {
        "id": 4,
        "title": "International Shipping Policy",
        "content": (
            "Acme Corp ships internationally to over 50 countries. International Standard "
            "Shipping takes 10-21 business days and costs vary by destination. International "
            "Express Shipping takes 5-7 business days. All international orders may be "
            "subject to customs duties, taxes, and import fees which are the responsibility "
            "of the customer. International returns must be shipped at the customer's expense, "
            "and refunds will not include the original shipping cost."
        ),
    },
    {
        "id": 5,
        "title": "Product Warranty Information",
        "content": (
            "All Acme Corp branded products come with a 1-year limited warranty covering "
            "defects in materials and workmanship under normal use. The warranty does not "
            "cover damage caused by accidents, misuse, unauthorized modifications, or normal "
            "wear and tear. To make a warranty claim, customers must provide proof of purchase "
            "and a description of the defect. Extended warranty plans (2-year and 3-year) are "
            "available for purchase at the time of original purchase for an additional fee."
        ),
    },
    {
        "id": 6,
        "title": "Customer Support Channels",
        "content": (
            "Acme Corp customer support is available through multiple channels. Phone support "
            "is available Monday through Friday, 8 AM to 8 PM EST at 1-800-ACME-HELP "
            "(1-800-226-3435). Email support can be reached at support@acmecorp.com with a "
            "typical response time of 24-48 hours. Live chat is available on our website "
            "Monday through Saturday, 9 AM to 6 PM EST. For urgent issues outside business "
            "hours, customers can use our automated help center at help.acmecorp.com."
        ),
    },
    {
        "id": 7,
        "title": "Product: Acme SmartHome Hub",
        "content": (
            "The Acme SmartHome Hub is our flagship home automation controller priced at "
            "$149.99. It supports WiFi, Bluetooth, Zigbee, and Z-Wave protocols, making it "
            "compatible with over 10,000 smart home devices. Features include voice control "
            "via built-in microphones, a 5-inch touchscreen display, energy monitoring "
            "dashboard, and automated routines. The SmartHome Hub comes with a 2-year warranty."
        ),
    },
    {
        "id": 8,
        "title": "Product: Acme AirPure Pro Air Purifier",
        "content": (
            "The Acme AirPure Pro is a premium air purifier designed for rooms up to 800 "
            "square feet, priced at $299.99. It features a 4-stage filtration system: "
            "pre-filter, activated carbon filter, True HEPA H13 filter, and UV-C light "
            "sanitizer. The AirPure Pro removes 99.97% of particles as small as 0.3 microns. "
            "Filter replacements are recommended every 6-8 months and cost $49.99 for the "
            "HEPA filter and $29.99 for the carbon filter."
        ),
    },
    {
        "id": 9,
        "title": "Accepted Payment Methods",
        "content": (
            "Acme Corp accepts the following payment methods: Visa, MasterCard, American "
            "Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. For "
            "orders over $200, we also offer Acme Pay Later, a buy-now-pay-later option that "
            "splits payments into 4 interest-free installments over 6 weeks. All payment "
            "information is encrypted using 256-bit SSL encryption."
        ),
    },
    {
        "id": 10,
        "title": "Acme Rewards Loyalty Program",
        "content": (
            "The Acme Rewards program is free to join and earns members 1 point per dollar "
            "spent. Points can be redeemed for discounts: 100 points equals $5 off your next "
            "purchase. Members also receive exclusive benefits including early access to sales, "
            "birthday discounts (20% off), and free standard shipping on all orders. Silver "
            "tier (500+ points/year) unlocks free expedited shipping. Gold tier (1000+ "
            "points/year) adds priority customer support and exclusive product previews. "
            "Points expire after 12 months of account inactivity."
        ),
    },
]

print(f"Loaded {len(ACME_DOCUMENTS)} documents")

In [ ]:
# Chunk the documents
# For this knowledge base the documents are already reasonably sized,
# so we treat each document as a single chunk.

chunks = []
for doc in ACME_DOCUMENTS:
    chunks.append({
        "id": doc["id"],
        "text": f"{doc['title']}\n\n{doc['content']}",
        "title": doc["title"],
    })

print(f"Created {len(chunks)} chunks")
for c in chunks:
    print(f"  [{c['id']}] {c['title']} ({len(c['text'])} chars)")

In [ ]:
# Embed all chunks using OpenAI
def get_embeddings(texts, model=EMBEDDING_MODEL):
    """Get embeddings for a list of texts."""
    response = openai_client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = get_embeddings(chunk_texts)

print(f"Generated {len(chunk_embeddings)} embeddings of dimension {len(chunk_embeddings[0])}")

In [ ]:
# Create Qdrant collection and upsert embeddings
qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=chunk["id"],
        vector=embedding,
        payload={"text": chunk["text"], "title": chunk["title"]},
    )
    for chunk, embedding in zip(chunks, chunk_embeddings)
]

qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Upserted {len(points)} vectors into Qdrant collection '{COLLECTION_NAME}'")

In [ ]:
# Build retriever function
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Retrieve top-k relevant chunks for a query."""
    query_embedding = get_embeddings([query])[0]
    results = qdrant_client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_embedding,
        limit=top_k,
    )
    return [hit.payload["text"] for hit in results]

# Build generator function
def generate(query: str, contexts: list[str]) -> str:
    """Generate an answer given a query and retrieved contexts."""
    context_str = "\n\n---\n\n".join(contexts)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful customer support assistant for Acme Corp. "
                "Answer questions based on the provided context. "
                "If the answer is not in the context, say so. Be concise and accurate."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context_str}\n\nQuestion: {query}",
        },
    ]
    response = openai_client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=messages,
        temperature=0.0,
    )
    return response.choices[0].message.content

# Full RAG pipeline
def rag_pipeline(query: str, top_k: int = 3) -> dict:
    """Run the full RAG pipeline: retrieve then generate."""
    contexts = retrieve(query, top_k=top_k)
    answer = generate(query, contexts)
    return {"query": query, "answer": answer, "contexts": contexts}

# Quick test
test_result = rag_pipeline("What is Acme Corp's return policy?")
print(f"Query: {test_result['query']}")
print(f"Answer: {test_result['answer']}")
print(f"Contexts retrieved: {len(test_result['contexts'])}")

## 3. Create the RAGAS Evaluation Dataset

RAGAS uses `SingleTurnSample` objects to represent individual evaluation samples. Each sample contains:
- `user_input`: The user's question
- `response`: The RAG system's answer
- `reference`: The ground-truth / expected answer
- `retrieved_contexts`: The contexts that were retrieved

We will create 12 test cases covering different question types and difficulty levels.

In [ ]:
# Define test cases: (query, reference_answer)
test_cases = [
    {
        "query": "What is the return policy for regular items?",
        "reference": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt. Refunds are processed in 5-7 business days.",
    },
    {
        "query": "How long do I have to return electronics?",
        "reference": "Electronics have a 15-day return window. They must be returned with all original accessories and packaging. A 15% restocking fee may apply to opened items. Defective electronics can be exchanged within 90 days.",
    },
    {
        "query": "What shipping options are available and how much do they cost?",
        "reference": "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited Shipping (2-3 days, $12.99), and Overnight Shipping (next business day, $24.99 if ordered before 2 PM EST).",
    },
    {
        "query": "Do you ship internationally?",
        "reference": "Yes, Acme Corp ships to over 50 countries. Standard international shipping takes 10-21 business days, Express takes 5-7 days. Customers are responsible for customs duties and import fees.",
    },
    {
        "query": "What does the warranty cover on Acme products?",
        "reference": "Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship. It does not cover accidents, misuse, or normal wear. Extended 2-year and 3-year plans are available.",
    },
    {
        "query": "How can I contact customer support?",
        "reference": "Customer support is available by phone (1-800-ACME-HELP, M-F 8AM-8PM EST), email (support@acmecorp.com, 24-48hr response), and live chat (M-Sat 9AM-6PM EST).",
    },
    {
        "query": "What are the features of the Acme SmartHome Hub?",
        "reference": "The SmartHome Hub costs $149.99, supports WiFi/Bluetooth/Zigbee/Z-Wave, has voice control, 5-inch touchscreen, energy monitoring, and automated routines. It comes with a 2-year warranty.",
    },
    {
        "query": "How much is the AirPure Pro and what does it filter?",
        "reference": "The AirPure Pro costs $299.99 and uses a 4-stage filtration system (pre-filter, carbon, True HEPA H13, UV-C). It removes 99.97% of particles 0.3 microns or larger, covers rooms up to 800 sq ft.",
    },
    {
        "query": "What payment methods do you accept?",
        "reference": "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. Orders over $200 qualify for Acme Pay Later (4 installments over 6 weeks).",
    },
    {
        "query": "How does the Acme Rewards program work?",
        "reference": "Acme Rewards is free, earning 1 point per dollar. 100 points = $5 off. Benefits include early sale access, birthday 20% discount, and free shipping. Silver tier (500+ pts) adds free expedited shipping. Gold tier (1000+ pts) adds priority support.",
    },
    {
        "query": "What is the electronics restocking fee?",
        "reference": "A restocking fee of 15% may apply to opened electronics returned to Acme Corp.",
    },
    {
        "query": "How long does it take to process a refund?",
        "reference": "Refunds are processed within 5-7 business days after Acme Corp receives the returned item.",
    },
]

print(f"Defined {len(test_cases)} test cases")

In [ ]:
# Run each test case through the RAG pipeline and collect results
evaluation_data = []

for tc in test_cases:
    result = rag_pipeline(tc["query"])
    evaluation_data.append({
        "query": tc["query"],
        "response": result["answer"],
        "reference": tc["reference"],
        "contexts": result["contexts"],
    })
    print(f"Processed: {tc['query'][:60]}...")

print(f"\nCollected {len(evaluation_data)} evaluation samples")

In [ ]:
# Preview one sample
sample = evaluation_data[0]
print("=" * 70)
print(f"Query: {sample['query']}")
print(f"\nResponse: {sample['response']}")
print(f"\nReference: {sample['reference']}")
print(f"\nRetrieved Contexts ({len(sample['contexts'])}):\n")
for i, ctx in enumerate(sample["contexts"]):
    print(f"  Context {i+1}: {ctx[:120]}...")
print("=" * 70)

In [ ]:
# Create RAGAS SingleTurnSample objects and EvaluationDataset
samples = []
for ed in evaluation_data:
    sample = SingleTurnSample(
        user_input=ed["query"],
        response=ed["response"],
        reference=ed["reference"],
        retrieved_contexts=ed["contexts"],
    )
    samples.append(sample)

eval_dataset = EvaluationDataset(samples=samples)
print(f"Created EvaluationDataset with {len(eval_dataset)} samples")

## 4. Configure RAGAS LLM and Embeddings

RAGAS uses LLM-as-a-judge for most metrics. We need to wrap our LLM and embeddings
using LangChain wrappers so RAGAS can use them.

In [ ]:
# Configure the evaluator LLM and embeddings for RAGAS
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

print("Evaluator LLM and embeddings configured.")

## 5. Faithfulness

### How Faithfulness Works

Faithfulness measures whether the generated answer is **grounded in the retrieved context**. A faithful answer only contains claims that can be verified from the provided context.

**Algorithm (Claim Extraction + NLI)**:
1. **Claim Extraction**: The LLM extracts individual factual claims from the generated answer
2. **Natural Language Inference (NLI)**: Each claim is checked against the retrieved contexts to determine if it is supported (entailed), contradicted, or neither
3. **Score Calculation**: `Faithfulness = (Number of supported claims) / (Total number of claims)`

A score of 1.0 means every claim in the answer is supported by the context. A score of 0.0 means no claims are supported.

In [ ]:
# Run Faithfulness metric
faithfulness_metric = Faithfulness(llm=evaluator_llm)

faithfulness_results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness_metric],
)

faithfulness_df = faithfulness_results.to_pandas()
print("Faithfulness Scores:")
print("=" * 60)
for idx, row in faithfulness_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row['faithfulness']:.3f}")
print(f"\nMean Faithfulness: {faithfulness_df['faithfulness'].mean():.3f}")

### Interpreting Faithfulness Scores

| Score Range | Interpretation |
|-------------|----------------|
| 0.9 - 1.0  | Highly faithful -- all claims grounded in context |
| 0.7 - 0.9  | Mostly faithful -- minor unsupported claims |
| 0.5 - 0.7  | Partially faithful -- significant unsupported content |
| 0.0 - 0.5  | Low faithfulness -- mostly hallucinated |

Low faithfulness indicates the LLM is adding information not present in the retrieved context (hallucination).

## 6. Answer Relevancy

### How Answer Relevancy Works

Answer Relevancy measures whether the generated answer is **relevant to the user's question**. An answer can be faithful (grounded) but still irrelevant if it does not address what was asked.

**Algorithm (Reverse Question Generation + Embedding Similarity)**:
1. **Question Generation**: Given the answer, the LLM generates N hypothetical questions that the answer could be responding to
2. **Embedding Similarity**: Each generated question is compared to the original question using embedding cosine similarity
3. **Score Calculation**: `Answer Relevancy = mean(cosine_similarity(original_question, generated_question_i))` for all i

A score of 1.0 means the answer perfectly addresses the question. Low scores indicate the answer is off-topic or only partially relevant.

In [ ]:
# Run Answer Relevancy metric
relevancy_metric = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)

relevancy_results = evaluate(
    dataset=eval_dataset,
    metrics=[relevancy_metric],
)

relevancy_df = relevancy_results.to_pandas()
print("Answer Relevancy Scores:")
print("=" * 60)
for idx, row in relevancy_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row['answer_relevancy']:.3f}")
print(f"\nMean Answer Relevancy: {relevancy_df['answer_relevancy'].mean():.3f}")

## 7. Context Precision

### How Context Precision Works

Context Precision evaluates whether the **relevant chunks appear at the top** of the retrieved results. It measures the signal-to-noise ratio of retrieved contexts with respect to the reference answer.

**Algorithm**:
1. For each retrieved context chunk, the LLM judges whether it is relevant (useful for answering the question given the reference)
2. Precision is calculated at each rank position k: `precision@k = (relevant items in top k) / k`
3. **Score**: Average precision weighted by relevance at each position

High context precision means relevant documents were ranked first. Low context precision means relevant documents are buried among irrelevant ones.

In [ ]:
# Run Context Precision metric
context_precision_metric = LLMContextPrecisionWithReference(llm=evaluator_llm)

context_precision_results = evaluate(
    dataset=eval_dataset,
    metrics=[context_precision_metric],
)

ctx_prec_df = context_precision_results.to_pandas()
print("Context Precision Scores:")
print("=" * 60)
for idx, row in ctx_prec_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row['llm_context_precision_with_reference']:.3f}")
print(f"\nMean Context Precision: {ctx_prec_df['llm_context_precision_with_reference'].mean():.3f}")

## 8. Context Recall

### How Context Recall Works

Context Recall measures whether **all the information needed to answer the question** was actually retrieved. It checks if the ground truth answer can be attributed to the retrieved contexts.

**Algorithm**:
1. The reference (ground-truth) answer is broken into individual sentences/claims
2. For each sentence in the reference, the LLM checks if it can be attributed to any retrieved context
3. **Score**: `Context Recall = (Number of reference sentences attributable to context) / (Total reference sentences)`

A score of 1.0 means the retrieved contexts contain all information in the reference answer. Low recall means important information was not retrieved.

In [ ]:
# Run Context Recall metric
context_recall_metric = LLMContextRecall(llm=evaluator_llm)

context_recall_results = evaluate(
    dataset=eval_dataset,
    metrics=[context_recall_metric],
)

ctx_recall_df = context_recall_results.to_pandas()
print("Context Recall Scores:")
print("=" * 60)
for idx, row in ctx_recall_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row['llm_context_recall']:.3f}")
print(f"\nMean Context Recall: {ctx_recall_df['llm_context_recall'].mean():.3f}")

## 9. Context Entity Recall

### How Context Entity Recall Works

Context Entity Recall is a complementary metric to regular Context Recall. Instead of checking sentence-level attribution, it focuses on **named entities** (people, organizations, numbers, dates, etc.).

**What it adds over regular recall**:
- Regular recall checks if sentences from the reference can be attributed to context
- Entity recall checks if specific **facts** (entities) mentioned in the reference appear in the retrieved context
- This catches cases where the context paraphrases information but misses specific entities
- It is a non-LLM metric -- purely based on entity extraction and comparison

**Algorithm**:
1. Extract named entities from the reference answer
2. Extract named entities from the retrieved contexts
3. **Score**: `Entity Recall = |entities in reference AND context| / |entities in reference|`

In [ ]:
# Run Context Entity Recall metric
entity_recall_metric = ContextEntityRecall(llm=evaluator_llm)

entity_recall_results = evaluate(
    dataset=eval_dataset,
    metrics=[entity_recall_metric],
)

entity_recall_df = entity_recall_results.to_pandas()
print("Context Entity Recall Scores:")
print("=" * 60)
for idx, row in entity_recall_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row['context_entity_recall']:.3f}")
print(f"\nMean Context Entity Recall: {entity_recall_df['context_entity_recall'].mean():.3f}")

## 10. Answer Correctness

### How Answer Correctness Works

Answer Correctness evaluates the **overall quality** of the generated answer by combining two components:

1. **Factual Correctness (F1 Score)**: Compares claims in the generated answer against the reference answer
   - True Positive (TP): Claims present in both answer and reference
   - False Positive (FP): Claims in answer but not in reference
   - False Negative (FN): Claims in reference but not in answer
   - `F1 = 2 * TP / (2 * TP + FP + FN)`

2. **Semantic Similarity**: Embedding-based similarity between the answer and reference

The final score is a weighted combination:
`Answer Correctness = w1 * Factual_Correctness + w2 * Semantic_Similarity`

By default, the weights are 0.75 for factual and 0.25 for semantic.

In [ ]:
# Run Factual Correctness and Semantic Similarity metrics
factual_correctness_metric = FactualCorrectness(llm=evaluator_llm)
semantic_similarity_metric = SemanticSimilarity(embeddings=evaluator_embeddings)

correctness_results = evaluate(
    dataset=eval_dataset,
    metrics=[factual_correctness_metric, semantic_similarity_metric],
)

correctness_df = correctness_results.to_pandas()
print("Answer Correctness Scores:")
print("=" * 80)
print(f"  {'Query':<45} {'Factual':>10} {'Semantic':>10}")
print("-" * 80)
for idx, row in correctness_df.iterrows():
    query_short = row["user_input"][:43]
    factual = row.get("factual_correctness", 0)
    semantic = row.get("semantic_similarity", 0)
    print(f"  [{idx+1:2d}] {query_short:<43} {factual:>8.3f}   {semantic:>8.3f}")
print(f"\nMean Factual Correctness: {correctness_df['factual_correctness'].mean():.3f}")
print(f"Mean Semantic Similarity:  {correctness_df['semantic_similarity'].mean():.3f}")

## 11. Run ALL Metrics Together

Now let us run all metrics in a single `evaluate()` call. This is the recommended approach for production use, as it is more efficient and provides a complete picture.

In [ ]:
# Define all metrics
all_metrics = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    ContextEntityRecall(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm),
    SemanticSimilarity(embeddings=evaluator_embeddings),
]

print(f"Running {len(all_metrics)} metrics on {len(eval_dataset)} samples...")
print("This may take a few minutes.\n")

full_results = evaluate(
    dataset=eval_dataset,
    metrics=all_metrics,
)

print("Evaluation complete!")
print(f"\nAggregate scores: {full_results}")

In [ ]:
# Display full results as a DataFrame
results_df = full_results.to_pandas()

# Select and rename key columns for display
metric_columns = [
    "faithfulness",
    "answer_relevancy",
    "llm_context_precision_with_reference",
    "llm_context_recall",
    "context_entity_recall",
    "factual_correctness",
    "semantic_similarity",
]

display_columns = {
    "faithfulness": "Faithfulness",
    "answer_relevancy": "Ans Relevancy",
    "llm_context_precision_with_reference": "Ctx Precision",
    "llm_context_recall": "Ctx Recall",
    "context_entity_recall": "Entity Recall",
    "factual_correctness": "Factual Corr",
    "semantic_similarity": "Semantic Sim",
}

# Create a clean display DataFrame
display_df = results_df[["user_input"] + metric_columns].copy()
display_df.columns = ["Query"] + [display_columns[c] for c in metric_columns]
display_df["Query"] = display_df["Query"].str[:50]

# Show the full results table
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.3f}".format)
display_df

In [ ]:
# Summary statistics for all metrics
summary = results_df[metric_columns].describe()
summary.columns = [display_columns[c] for c in metric_columns]
print("Summary Statistics Across All Metrics:")
print("=" * 80)
summary.round(3)

In [ ]:
# Identify worst-performing queries for each metric
print("Worst-Performing Queries per Metric:")
print("=" * 80)

for col in metric_columns:
    worst_idx = results_df[col].idxmin()
    worst_score = results_df.loc[worst_idx, col]
    worst_query = results_df.loc[worst_idx, "user_input"][:60]
    friendly_name = display_columns[col]
    print(f"\n  {friendly_name}:")
    print(f"    Worst query: \"{worst_query}\"")
    print(f"    Score: {worst_score:.3f}")

In [ ]:
# Compare scores across test cases using a bar chart
import matplotlib
matplotlib.use("Agg")  # Use non-interactive backend
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top chart: Retriever metrics
retriever_metrics = ["llm_context_precision_with_reference", "llm_context_recall", "context_entity_recall"]
retriever_names = ["Ctx Precision", "Ctx Recall", "Entity Recall"]
x = np.arange(len(results_df))
width = 0.25

for i, (metric, name) in enumerate(zip(retriever_metrics, retriever_names)):
    axes[0].bar(x + i * width, results_df[metric], width, label=name)

axes[0].set_xlabel("Test Case")
axes[0].set_ylabel("Score")
axes[0].set_title("Retriever Metrics by Test Case")
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([f"Q{i+1}" for i in range(len(results_df))], rotation=0)
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# Bottom chart: Generator metrics
generator_metrics = ["faithfulness", "answer_relevancy", "factual_correctness", "semantic_similarity"]
generator_names = ["Faithfulness", "Ans Relevancy", "Factual Corr", "Semantic Sim"]
width = 0.2

for i, (metric, name) in enumerate(zip(generator_metrics, generator_names)):
    axes[1].bar(x + i * width, results_df[metric], width, label=name)

axes[1].set_xlabel("Test Case")
axes[1].set_ylabel("Score")
axes[1].set_title("Generator Metrics by Test Case")
axes[1].set_xticks(x + 1.5 * width)
axes[1].set_xticklabels([f"Q{i+1}" for i in range(len(results_df))], rotation=0)
axes[1].legend()
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig("ragas_metrics_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("Chart saved to ragas_metrics_comparison.png")

In [ ]:
# Correlation analysis between metrics
correlation_matrix = results_df[metric_columns].corr()
correlation_matrix.columns = [display_columns[c] for c in metric_columns]
correlation_matrix.index = [display_columns[c] for c in metric_columns]

print("Metric Correlation Matrix:")
print("=" * 80)
print(correlation_matrix.round(3))

## 12. Interpretation Guide

### What Each Metric Tells You

| Metric | Measures | Component | Action if Low |
|--------|----------|-----------|---------------|
| **Faithfulness** | Is the answer grounded in context? | Generator | Improve prompt to reduce hallucination |
| **Answer Relevancy** | Does the answer address the question? | Generator | Improve prompt instructions |
| **Context Precision** | Are relevant docs ranked high? | Retriever | Improve ranking/reranking |
| **Context Recall** | Was all needed info retrieved? | Retriever | Improve retrieval (more chunks, better embeddings) |
| **Entity Recall** | Were key entities retrieved? | Retriever | Check chunking strategy for entity coverage |
| **Factual Correctness** | Does answer match reference facts? | End-to-end | Investigate both retriever and generator |
| **Semantic Similarity** | Is answer semantically close to reference? | End-to-end | General quality indicator |

### Diagnostic Flow

1. **Low Context Recall + Low Context Precision** => Retriever problem. Fix chunking, embeddings, or add more documents.
2. **High Context Recall + Low Faithfulness** => Generator problem. The right context was retrieved but the LLM hallucinated.
3. **High Faithfulness + Low Answer Relevancy** => The answer is grounded but off-topic. Improve the system prompt.
4. **High everything + Low Factual Correctness** => The reference answers may need updating, or there is a subtle reasoning error.

### Recommended Thresholds

For production RAG systems:
- Faithfulness >= 0.85
- Answer Relevancy >= 0.80
- Context Precision >= 0.75
- Context Recall >= 0.80
- Factual Correctness >= 0.70

In [ ]:
# Final quality check: Flag any samples below thresholds
thresholds = {
    "faithfulness": 0.85,
    "answer_relevancy": 0.80,
    "llm_context_precision_with_reference": 0.75,
    "llm_context_recall": 0.80,
    "factual_correctness": 0.70,
}

print("Quality Check -- Flagged Samples:")
print("=" * 80)

flagged_count = 0
for idx, row in results_df.iterrows():
    issues = []
    for metric, threshold in thresholds.items():
        if row[metric] < threshold:
            friendly = display_columns[metric]
            issues.append(f"{friendly}={row[metric]:.2f} (< {threshold})")
    if issues:
        flagged_count += 1
        print(f"\n  Query: \"{row['user_input'][:60]}\"")
        for issue in issues:
            print(f"    WARNING: {issue}")

if flagged_count == 0:
    print("\n  All samples pass quality thresholds!")
else:
    print(f"\n  {flagged_count}/{len(results_df)} samples flagged for review.")

## Summary

In this notebook we covered the full suite of RAGAS core metrics:

1. **Faithfulness** -- Checks if the answer is grounded in the retrieved context (claim extraction + NLI)
2. **Answer Relevancy** -- Checks if the answer addresses the question (reverse question generation + embedding similarity)
3. **Context Precision** -- Checks if relevant documents are ranked high in retrieval results
4. **Context Recall** -- Checks if all needed information was retrieved
5. **Context Entity Recall** -- Checks entity-level coverage in retrieved contexts
6. **Factual Correctness** -- Compares factual claims between answer and reference
7. **Semantic Similarity** -- Embedding-based similarity between answer and reference

**Next notebook**: We will explore advanced RAGAS features including testset generation, custom metrics, and framework comparison.